# SPOT phase-transition simulation ($\alpha=0.5$)

This notebook reproduces the one-dimensional phase-transition slices. The current manuscript calls Algorithm 1 **SPOT**.

In [ ]:
import os
import math
import hashlib
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| Device:", DEVICE)

np.random.seed(0)

OUTPUT_DIR = "outputs_discovery_v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def seed_from_params(base_seed: int, **kwargs) -> int:
    items = ",".join(f"{key}={kwargs[key]}" for key in sorted(kwargs))
    value = f"{base_seed}|{items}"
    digest = hashlib.blake2b(value.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "little") & 0xFFFFFFFF


## Simulation model and SPOT parameters

In [ ]:
def sample_theta_iid(
    n: int,
    eps: float,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    return (rng.random(n) < eps).astype(np.int8)


def sample_theta_markov_stationary(
    n: int,
    eps: float,
    s: float = 0.6,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[np.ndarray, float, float]:
    if rng is None:
        rng = np.random.default_rng()
    s = float(np.clip(s, 1e-6, 1.0 - 1e-6))
    eps = float(np.clip(eps, 0.0, 1.0))
    a = eps * s
    b = (1.0 - eps) * s

    theta = np.zeros(n, dtype=np.int8)
    theta[0] = int(rng.random() < eps)
    for t in range(1, n):
        if theta[t - 1] == 0:
            theta[t] = int(rng.random() < a)
        else:
            theta[t] = int(rng.random() >= b)
    return theta, a, b


@dataclass
class SimParams:
    n: int
    alpha_vocab: float
    r_core: float
    p_sparse: float
    q_sing: float
    theta_mode: str = "markov"
    markov_s: float = 0.6
    kappa_core_mass: float = 0.2
    eps_C0: float = 0.5
    pt_mode: str = "hclplus_indepmix"
    oracle_eps: bool = False
    joint_rho: float = 0.95
    joint_amp: float = 0.2
    hcl_cC: float = 0.5
    hcl_CC: float = 2.0


def _markov_chain_Z(
    n: int,
    rho: float = 0.95,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    rho = float(np.clip(rho, 1e-6, 1.0 - 1e-6))
    Z = np.empty(int(n), dtype=np.int8)
    Z[0] = int(rng.random() < 0.5)
    u = rng.random(int(n) - 1)
    for t in range(1, int(n)):
        Z[t] = Z[t - 1] if u[t - 1] < rho else 1 - Z[t - 1]
    return Z


def _hclplus_constants(
    n: int,
    alpha: float,
    r_core: float,
    q: float,
    cC: float,
    CC: float,
    c_light_mult: float,
    c_core_mult: float,
) -> Dict[str, float]:
    mL = int(max(1, round(n ** alpha)))
    mC = int(max(1, round(n ** r_core)))

    p_light = float(np.clip(c_light_mult, 0.0, 10.0)) * n ** (-(alpha + q))
    p_core = float(np.clip(c_core_mult, cC, CC)) * n ** (-(r_core + q))

    Delta = float(n ** (-q))
    Delta_core = float(np.clip(mC * p_core, 0.0, 0.9 * Delta))
    Delta_light = float(np.clip(mL * p_light, 0.0, 0.9 * Delta))
    total = Delta_core + Delta_light
    if total <= 0:
        Delta_core = 0.0
        Delta_light = 0.0
    elif total > Delta:
        scale = Delta / total
        Delta_core *= scale
        Delta_light *= scale

    p_heavy = float(np.clip(1.0 - Delta_core - Delta_light, 0.0, 1.0))
    p_core_eff = Delta_core / mC if mC > 0 else 0.0
    p_light_eff = Delta_light / mL if mL > 0 else 0.0

    return {
        "mL": float(mL),
        "mC": float(mC),
        "Delta": Delta,
        "Delta_core": Delta_core,
        "Delta_light": Delta_light,
        "p_heavy": p_heavy,
        "p_core": float(p_core_eff),
        "p_light": float(p_light_eff),
    }


def _make_hclplus_consts_for_Z(params: SimParams, Z_val: int) -> Dict[str, float]:
    amp = float(params.joint_amp)
    z_sign = 2 * int(Z_val) - 1
    c_light_mult = float(np.clip(1.0 + 0.25 * amp * z_sign, 0.7, 1.3))
    c_core_mult = float(np.clip(0.6 + 0.20 * amp * z_sign, 0.3, 1.0))

    return _hclplus_constants(
        n=int(params.n),
        alpha=float(params.alpha_vocab),
        r_core=float(params.r_core),
        q=float(params.q_sing),
        cC=float(params.hcl_cC),
        CC=float(params.hcl_CC),
        c_light_mult=c_light_mult,
        c_core_mult=c_core_mult,
    )


def _sample_Pt_w_from_consts(
    consts: Dict[str, float],
    m: int,
    rng: np.random.Generator,
) -> np.ndarray:
    if m <= 0:
        return np.empty(0, dtype=float)

    u = rng.random(int(m))
    pH = float(consts["p_heavy"])
    Dc = float(consts["Delta_core"])
    pC = float(consts["p_core"])
    pL = float(consts["p_light"])

    out = np.empty(int(m), dtype=float)
    maskH = u < pH
    maskC = (~maskH) & (u < pH + Dc)
    out[maskH] = pH
    out[maskC] = pC
    out[~(maskH | maskC)] = pL
    return out


def generate_pivots_H1(
    params: SimParams,
    rng: Optional[np.random.Generator] = None,
):
    if rng is None:
        rng = np.random.default_rng()

    n = int(params.n)
    eps_n = float(params.eps_C0) * float(n ** (-float(params.p_sparse)))

    seed_theta = int(rng.integers(0, 2**32 - 1))
    rng_theta = np.random.default_rng(seed_theta)

    Z = _markov_chain_Z(n, rho=float(params.joint_rho), rng=rng)

    theta_mode = str(params.theta_mode).lower()
    if theta_mode == "iid":
        theta = sample_theta_iid(n, eps_n, rng=rng_theta)
        a = b = None
    elif theta_mode == "markov":
        theta, a, b = sample_theta_markov_stationary(
            n,
            eps_n,
            s=float(params.markov_s),
            rng=rng_theta,
        )
    else:
        raise ValueError("theta_mode must be 'iid' or 'markov'")

    U = rng.random(n)
    Y = U.copy()
    signal = theta == 1

    if np.any(signal):
        consts0 = _make_hclplus_consts_for_Z(params, 0)
        consts1 = _make_hclplus_consts_for_Z(params, 1)

        mask0 = signal & (Z == 0)
        if np.any(mask0):
            Pt0 = _sample_Pt_w_from_consts(consts0, int(mask0.sum()), rng)
            Y[mask0] = U[mask0] ** Pt0

        mask1 = signal & (Z == 1)
        if np.any(mask1):
            Pt1 = _sample_Pt_w_from_consts(consts1, int(mask1.sum()), rng)
            Y[mask1] = U[mask1] ** Pt1

    info = {
        "Z": Z,
        "eps_n": eps_n,
        "rho_Z": float(params.joint_rho),
        "theta_mode": theta_mode,
        "markov_s": float(params.markov_s) if theta_mode == "markov" else None,
        "theta_a": a,
        "theta_b": b,
        "regime_amp": float(params.joint_amp),
    }
    return Y, theta, info


@dataclass
class Alg1Params:
    # Algorithm 1 is named SPOT in the current manuscript.
    umin: float = 0.005
    umax: float = 0.98
    tau_dagger: float = 0.85
    Mn_power: float = 1.5
    Mn_mult: float = 3.5
    eta_mult: float = 0.0
    eps_clip: float = 1e-8


## GPU evaluation and calibration over $C$

In [ ]:
EPS_RATIO = 1e-12


def eval_err_over_C_torch(
    params: SimParams,
    alg: Alg1Params,
    C_grid: np.ndarray,
    B: int = 200,
    rng: Optional[np.random.Generator] = None,
):
    if rng is None:
        rng = np.random.default_rng()

    n = int(params.n)
    logn = float(np.log(max(3, n)))
    M = int(max(10, math.ceil(alg.Mn_mult * logn ** alg.Mn_power)))
    U = np.linspace(float(alg.umin), float(alg.umax), M).astype(np.float64)
    taus = torch.tensor(1.0 - n ** (-U), device=DEVICE, dtype=torch.float64)

    Ys = []
    thetas = []
    eps_true = float(params.eps_C0) * float(n ** (-float(params.p_sparse)))
    for _ in range(int(B)):
        Y, theta, _ = generate_pivots_H1(params, rng=rng)
        Ys.append(Y.astype(np.float64, copy=False))
        thetas.append(theta.astype(np.int8, copy=False))

    Y_t = torch.tensor(np.stack(Ys), device=DEVICE, dtype=torch.float64)
    th_t = torch.tensor(np.stack(thetas), device=DEVICE, dtype=torch.int8)
    th_bool = th_t.bool()
    Y_sorted, _ = torch.sort(Y_t, dim=1)

    expanded_taus = taus.view(1, -1).expand(Y_sorted.shape[0], -1).contiguous()
    idx = torch.searchsorted(Y_sorted, expanded_taus, right=True)
    Shat = (n - idx).to(torch.float64) / float(n)
    S0 = (1.0 - taus).unsqueeze(0)

    tau_d = float(alg.tau_dagger)
    tau_d_t = torch.full((Y_sorted.shape[0], 1), tau_d, device=DEVICE, dtype=torch.float64)
    idx_d = torch.searchsorted(Y_sorted, tau_d_t, right=True).squeeze(1)
    Shat_d = (n - idx_d).to(torch.float64) / float(n)
    S0_d = 1.0 - tau_d
    eps_hat = (Shat_d - S0_d) / max(1e-15, 1.0 - S0_d)
    eps_hat = torch.clamp(eps_hat, 0.0, 1.0).to(torch.float64)

    mae_eps = torch.mean(torch.abs(eps_hat - eps_true)).item()
    std_eps = torch.std(eps_hat, unbiased=False).item()
    std_err = torch.std(eps_hat - eps_true, unbiased=False).item()

    denom = torch.maximum(Shat, torch.full_like(Shat, 1.0 / float(n)))
    That = (1.0 - eps_hat.unsqueeze(1)) * (S0 / denom)
    eta = float(alg.eta_mult) / logn**2

    curve = []
    best_err = float("inf")
    best_C = float(C_grid[0])

    for C in C_grid:
        cond = That <= float(C) / logn - eta
        any_hit = cond.any(dim=1)
        first_idx = torch.argmax(cond.to(torch.int8), dim=1)
        selected_idx = torch.where(any_hit, first_idx, torch.full_like(first_idx, M - 1))

        tau_hat = taus[selected_idx]
        delta = Y_t > tau_hat.unsqueeze(1)

        TP = (delta & th_bool).sum().item()
        FP = (delta & (~th_bool)).sum().item()
        FPR = float(FP) / float(TP + FP + EPS_RATIO)
        p_miss = (delta.sum(dim=1) == 0).to(torch.float64).mean().item()
        error = FPR + p_miss

        curve.append((float(C), float(error), float(FPR), float(p_miss)))
        if error < best_err - 1e-12:
            best_err = error
            best_C = float(C)

    return best_err, best_C, curve, mae_eps, std_eps, std_err


## Run the $\alpha=0.5$ phase-transition slices

In [ ]:
BASE_SEED = 20260123

C_grid = np.round(np.arange(0.05, 2.0001, 0.05), 2)
fixed_q = 0.4
fixed_p = 0.6
p_grid = np.linspace(0.05, 0.95, 19)
q_grid = np.linspace(0.05, 0.95, 19)
n_list = [100, 1000, 10000]

# Change only this value to reproduce the other alpha panels.
alpha_vocab = 0.5
r_core = 0.0
B = 200

alg = Alg1Params()


def run_slice_fix_q_vary_p(n: int, q_fixed: float) -> np.ndarray:
    out = []
    for p in p_grid:
        params = SimParams(
            n=int(n),
            alpha_vocab=float(alpha_vocab),
            r_core=float(r_core),
            p_sparse=float(p),
            q_sing=float(q_fixed),
            theta_mode="markov",
            markov_s=0.6,
            kappa_core_mass=0.2,
            eps_C0=0.5,
            pt_mode="hclplus_indepmix",
            joint_rho=0.95,
        )
        seed = seed_from_params(
            BASE_SEED,
            n=n,
            alpha=alpha_vocab,
            r=r_core,
            p=p,
            q=q_fixed,
        )
        rng = np.random.default_rng(seed)
        result = eval_err_over_C_torch(params, alg, C_grid=C_grid, B=B, rng=rng)
        min_err, C_star, _, mae_eps, std_eps, std_err = result
        out.append((p, min_err, C_star, mae_eps, std_eps, std_err))
        print(f"[q={q_fixed}] n={n}, p={p:.3f}: min error={min_err:.4f}, C*={C_star:.2f}")
    return np.asarray(out, dtype=float)


def run_slice_fix_p_vary_q(n: int, p_fixed: float) -> np.ndarray:
    out = []
    for q in q_grid:
        params = SimParams(
            n=int(n),
            alpha_vocab=float(alpha_vocab),
            r_core=float(r_core),
            p_sparse=float(p_fixed),
            q_sing=float(q),
            theta_mode="markov",
            markov_s=0.6,
            kappa_core_mass=0.2,
            eps_C0=0.5,
            pt_mode="hclplus_indepmix",
            joint_rho=0.95,
        )
        seed = seed_from_params(
            BASE_SEED,
            n=n,
            alpha=alpha_vocab,
            r=r_core,
            p=p_fixed,
            q=q,
        )
        rng = np.random.default_rng(seed)
        result = eval_err_over_C_torch(params, alg, C_grid=C_grid, B=B, rng=rng)
        min_err, C_star, _, mae_eps, std_eps, std_err = result
        out.append((q, min_err, C_star, mae_eps, std_eps, std_err))
        print(f"[p={p_fixed}] n={n}, q={q:.3f}: min error={min_err:.4f}, C*={C_star:.2f}")
    return np.asarray(out, dtype=float)


results_fixq = {n: run_slice_fix_q_vary_p(n, fixed_q) for n in n_list}
results_fixp = {n: run_slice_fix_p_vary_q(n, fixed_p) for n in n_list}


## Plot and save the figure as PDF

In [ ]:
plt.rcParams.update({"xtick.labelsize": 14, "ytick.labelsize": 14})


def add_boundary(ax, x: float, label: str) -> None:
    ax.axvline(x, linestyle="--", linewidth=1.5, alpha=0.85, color="tab:gray")
    ymin, ymax = ax.get_ylim()
    ax.text(
        x,
        ymax - 0.02 * (ymax - ymin),
        label,
        rotation=90,
        va="top",
        ha="right",
        fontsize=12,
        color="tab:gray",
        alpha=0.95,
    )


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for n in n_list:
    arr = results_fixq[n]
    line = ax.plot(arr[:, 0], arr[:, 1], label=f"n={n}", linewidth=2)[0]
    ax.plot(
        arr[:, 0],
        arr[:, 1],
        linestyle="None",
        marker="o",
        markersize=3.5,
        alpha=0.45,
        color=line.get_color(),
    )
ax.set_title(r"fix $q=0.4$, vary $p$", fontsize=18)
ax.set_xlabel("p", fontsize=16)
ax.set_ylabel(r"$\min_C\,\mathrm{Err}(C)$", fontsize=16)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=15)
p_boundary = float(min(alpha_vocab, 1.0 - fixed_q))
if ax.get_xlim()[0] <= p_boundary <= ax.get_xlim()[1]:
    add_boundary(ax, p_boundary, r"p = min{α, 1−q}")

ax = axes[1]
for n in n_list:
    arr = results_fixp[n]
    line = ax.plot(arr[:, 0], arr[:, 1], label=f"n={n}", linewidth=2)[0]
    ax.plot(
        arr[:, 0],
        arr[:, 1],
        linestyle="None",
        marker="o",
        markersize=3.5,
        alpha=0.45,
        color=line.get_color(),
    )
ax.set_title(r"fix $p=0.6$, vary $q$", fontsize=18)
ax.set_xlabel("q", fontsize=16)
ax.set_ylabel(r"$\min_C\,\mathrm{Err}(C)$", fontsize=16)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=15)
if fixed_p < alpha_vocab:
    q_boundary = float(1.0 - fixed_p)
    if ax.get_xlim()[0] <= q_boundary <= ax.get_xlim()[1]:
        add_boundary(ax, q_boundary, r"q = 1−p")

fig.tight_layout()
alpha_tag = str(alpha_vocab).replace(".", "p")
pdf_path = os.path.join(OUTPUT_DIR, f"phase_transition_alpha{alpha_tag}.pdf")
fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
print("Saved:", pdf_path)
plt.show()
